# PHARVO-beta: Dashboard Load Test

**Objective:** Verify that an authorized user (`rafi`) can successfully log in and that the primary **PHARVO Dashboard shell** (header, sidebar navigation, and main dashboard container) loads completely without errors.

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Credentials ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print("[INFO] Starting PHARVO Dashboard Load Test...")

    # Step 2: Open target PHARVO page
    driver.get(f"{BASE_URL}/")

    # Step 3: Find login inputs
    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    # Step 4: Enter credentials
    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    # Step 5: Perform the login action
    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 6: Wait for the expected Dashboard shell elements to appear
    # 6.1 Wait for the main page header to display "Dashboard"
    dashboard_heading = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//header//h1[contains(text(), 'Dashboard')]"))
    )

    # 6.2 Verify the sidebar navigation shell is present and displays the active Dashboard tab
    active_nav_item = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//aside//button[@aria-current='page' and contains(., 'Dashboard')]")
        )
    )

    # 6.3 Verify the dashboard container card loads (contains 'PHARVO Pharmacy')
    dashboard_card = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//*[contains(text(), 'PHARVO Pharmacy')]")
        )
    )

    # Step 7 & 8: Verify the result and print PASS/FAIL information
    if dashboard_heading.is_displayed() and active_nav_item.is_displayed():
        print(f"PASS: Dashboard shell loaded successfully for user '{USERNAME}'.")
        print(f"      - Page Heading: '{dashboard_heading.text}'")
        print(f"      - Active Navigation: '{active_nav_item.text.strip()}'")
        print(f"      - Pharmacy Banner: '{dashboard_card.text}'")
    else:
        print("FAIL: Dashboard shell elements were located but are not visible.")

except Exception as error:
    print(f"FAIL: Dashboard load test failed with error: {error}")

finally:
    # Step 9: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
